In [ ]:
import numpy as np
from matplotlib.image import imread
import matplotlib.pyplot as plt
import wave
import scipy.signal

# 1 Discrete Fourier Tranformation
## 1.1 (4 points)
To start with the analysis, implement the function `discrete_ft`, which computes the discrete Fourier transformation of a time-dependent function $x(t)$ and `discrete_ift`, which computes the inverse discrete Fourier transformation of a frequency-dependent function $\tilde{x}(\omega)$. The transformations are given by

$$
\tilde{x}_k = \sum\limits_{n=0}^{N-1} e^{-i \frac{2\pi kn}{N}} x_n,
$$ 

and

$$ 
x_n = \frac{1}{N} \sum\limits_{k=0}^{N-1} e^{i \frac{2\pi kn}{N}} \tilde{x}_k.
$$
Note that the definitions differ slightly from the ones given in the lecture in order to simplify the comparison with the numpy function.

In [ ]:
def discrete_ft(signal):
    """Computes the discrete Fourier Transformation of x.
    signal : 1-d array
        sequence to transform
    returns : 1-d array
        Fourier transformed signal
    """
    ### BEGIN SOLUTION
    N = len(signal)
    spectrum = np.zeros(N, dtype=np.complex128)
    for k in range(N):
        spectrum[k] = np.sum(signal * np.exp(-2j * np.pi * k * np.arange(N) / N))
    return spectrum
    ### END SOLUTION

def discrete_ift(spectrum):
    """Computes the inverser discrete Fourier Transformation of x.
    spectrum : 1-d array
        sequence to transform
    returns : 1-d array
        inverse Fourier transformed spectrum
    """
    ### BEGIN SOLUTION
    N = len(spectrum)
    signal = np.zeros(N, dtype=np.complex128)
    for n in range(N):
        signal[n] = np.sum(spectrum * np.exp(2j * np.pi * n * np.arange(N) / N)) / N
    return signal
    ### END SOLUTION

In [ ]:
t_test = np.linspace(0, 2*np.pi, 1000)
cos_test = 2 * np.sin(t_test) + 1j * np.cos(t_test)
cos_test_ft = discrete_ft(cos_test)
np.testing.assert_allclose(cos_test_ft, np.fft.fft(cos_test))
np.testing.assert_allclose(discrete_ift(cos_test_ft), np.fft.ifft(cos_test_ft))

# 2 Aliasing 
## 2.1 (4 points)
When sampling periodic data with frequency $\omega_\mathrm{signal}$ at discrete points with sampling frequency $\omega_\mathrm{sample}$ a reconstruction of the frequency of the signal is only possible up to a certain ambiguity. When the data is sampled at a frequency of $2 \omega_\mathrm{signal}$ or higher, the so-called Nyquist frequency, the signal can be fully reconstructed. At lower 
sampling frequencies, aliasing effects are encountered.     
To visualize this, plot the signal $s(t) = \sin(\omega_\mathrm{signal} t)$ with the angular frequency $\omega_\mathrm{signal} = 20$ over the time interval of $4 \pi$. Choose four different sampling frequencies below the Nyquist frequency and confirm that the apparent frequency of the signal does change by plotting the discretized sample alongside the original one in a single plot.    

*Tip: To increase the visibility, change the opacity of the signal with the highest sampling frequency by using `plt.plot(time, signal, alpha=0.3, ... )`*

In [ ]:
omega_signal = 20
t_total = 4 * np.pi
num_sampling_points = [600, 100, 40, 20, 10]

### BEGIN SOLUTION
for n in num_sampling_points:
    omega_sample = (n - 1) / t_total
    time = np.linspace(0, t_total, n)
    if n >= 500:
        plt.plot(time, np.sin(20 * time), label=f"{omega_sample:.2f}", alpha=0.3)
    else:
        plt.plot(time, np.sin(20 * time), label=f"{omega_sample:2f}")
plt.legend(title=r"$\omega_{sample}$")
plt.xlabel("t")
plt.ylabel("f(t)")
### END SOLUTION

# 2.2 (4 points)
The effect occurs because the family of functions $\{\sin\left(\left( \omega_\mathrm{signal} - N \omega_\mathrm{sample} \right) t \right), N \in \mathbb{Z} \}$ becomes indistinguishable when sampled at a frequency $\omega_\mathrm{sample}$.
Verify this by plotting $\sin(t)$ over an interval $[0, 16\pi]$ with $7$ sample points, corresponding to a sample frequency of  $\omega_\mathrm{sample} = 3/4$.
Mark the sampling points in the plot. Plot the function $\sin(t)$ at a much higher sample frequency.
When sampling oscillating data at a given sampling frequency, the can appear to have a lower frequency than the original one. The apparent frequency $\omega_{a}$ results from $\omega_{a} = \min_{N \in \mathbb{Z}}(\left| \omega_\mathrm{signal} - N \omega_\mathrm{sample} \right|)$. Which apparent frequency $\omega_{a}$ do you obtain here? Plot the apparent sinus sinusoidal the same plot as the sampling points.

In [ ]:
t_aliasing = np.arange(7) * 2 * np.pi * 4 / 3
t_plot = np.linspace(0, 16*np.pi, num=2001, endpoint=False)
sin_aliasing = np.sin(t_aliasing)

### BEGIN SOLUTION
# omega_signal = 1
# omega_samp = 3/4
# => |omega_signal - N omega_samp| = |1 - 3N / 4|
# which is minimized by N = 1
# therefore omega_a = |1 - 3/4| = 1/4

plt.plot(t_aliasing, sin_aliasing, 'o')
plt.xlabel("t")
plt.ylabel(r"$\sin(\omega\,t)$")
# signals
sin1 = np.sin(t_plot)
sin2 = np.sin(1 / 4 * t_plot) # omega_a = 1/4
plt.plot(t_plot, sin1, label=r"$\omega_{a}=1$")
plt.plot(t_plot, sin2, label=r"$\omega_{a}=1/4$")
plt.legend(loc="lower right")
### END SOLUTION

plt.show()


# 3 Sound analysis
## 3.1 (2 points)
We provide you with a sound sample of a C3 piano note. The sampling frequency of the .wav file is 44.1 kHz. After reading the file, we have a `signal` as a `numpy.array`. As the array is pretty large, it is recommended to use the functions `np.fft.fft` and `np.fft.ifft` for your Fourier transformations. We use the  NumPy's `np.fft.fftfreq` to generate the frequencies corresponding to the output of `np.fft.fft` and `np.fft.fftshift` to center the zero frequency. Plot the signal as a function of time and its frequency representation. In the frequency representation, it is useful to plot the data in a log-log scale with `xmin = 10` Hz and a suitably chosen `ymin` for the amplitude, such that the pronounced maxima are clearly visible.

In [ ]:
import IPython.display as ipd
ipd.Audio('./C3_131_Hz.wav')

In [ ]:
def readWave(file):
    with wave.open(file, 'r') as spf:
        signal = spf.readframes(-1)
        signal = np.frombuffer(signal, "int16")
        sampling_freq = spf.getframerate()  # 1/s
        return signal,sampling_freq

def writeWave(file,signal):
    with wave.open(file, 'wb') as spf:
        spf.setframerate(sampling_freq)
        spf.setnchannels(1)
        spf.setsampwidth(2)
        spf.writeframes(np.int16(signal))

signal,sampling_freq = readWave('./C3_131_Hz.wav')

In [ ]:
t = np.arange(len(signal)) / sampling_freq
dt = t[1] - t[0]
signal_f = np.fft.fftshift(np.fft.fft(signal))
freqs = np.fft.fftshift(np.fft.fftfreq(len(signal)) / dt)

### BEGIN SOLUTION
plt.plot(t, signal)
plt.xlabel("t / s")
plt.ylabel("amplitude")
plt.show()

plt.loglog(freqs, np.abs(signal_f))
plt.ylim(1e4)
plt.xlim(10)
plt.xlabel("f / Hz")
plt.ylabel("amplitude")
plt.show()
### END SOLUTION

## 3.2 (2 points)
The ground frequency of a C3 is supposed to be around 131 Hz. You should see the corresponding peak in the spectrum. For the specific piano that was recorded, the ground frequency can deviate a little. Use `scipy.signal.find_peaks` to find the ground frequency and the first 6 dominant overtones of the note. In theory those should be roughly integer multiples of the ground frequency.

In [ ]:
### BEGIN SOLUTION
peaks, _ =scipy.signal.find_peaks(np.abs(signal_f[len(signal)//2:]), height = 2e4, distance = 500)
peak_f = freqs[len(signal)//2:][peaks][1:8]
print(peak_f)
print('multiples of 131: ', 131 * (1 + np.arange(7)))
### END SOLUTION

## 3.3 (1 point) — Piano Sound Synthesis - Ground Tone

In this task, you will synthesize a piano-like sound signal. First, we generate a pure tone for 
the C3 note, which has a ground frequency of $\omega_0 = 131$ Hz.  
Generate a pure cosine signal
$$
x_0(t) = A \cos(\omega_0 t),
$$
select a suitable amplitude $A$
and create an audio signal of approximately $T = 4\,\mathrm{s}$ duration by sampling $x_0(t)$ at the same sampling rate  
as in the task above.

In [ ]:
### BEGIN SOLUTION
T = 4 # period in seconds
t_target = np.arange(T * sampling_freq) / sampling_freq # time series
freq_target = 131 # periods per second
omega_target = freq_target * 2 * np.pi # radians per second
signal_target = 20000 * np.cos(omega_target * t_target) # cosine with freq omega_target
### END SOLUTION
writeWave('ground.wav', signal_target)
ipd.Audio('./ground.wav')

## 3.4 (1 point) — Piano Sound Synthesis - Overtones

Next, construct a synthetic piano sound by adding the first six overtones (harmonics) of the ground frequency. The signal is modeled as

$$
x(t) = \sum_{k=1}^{7} A_k \cos(k \omega_0 t),
$$

where, $k=1$ corresponds to the ground tone, $k=2,\dots,7$ are the first six overtones, and $A_k$ are the relative amplitudes.
Estimate the amplitude ratios $A_k/A_1$ from a recorded C3 piano note and use these ratios to determine the relative heights of the harmonics.

Construct the corresponding Fourier spectrum with peaks at frequencies $k \omega_0$ and amplitudes $A_k$, and apply an inverse Fourier transform to obtain a time-domain signal of duration $T = 4\,\mathrm{s}$.


In [ ]:
### BEGIN SOLUTION
freq_target = np.fft.fftfreq(len(t_target)) / (t_target[1] - t_target[0])
ft_target = np.zeros(len(t_target), dtype=complex)
# for freq  in 131 * (2 + np.arange(5)):
for freq, peak_idx in zip(peak_f, peaks):
    idx_pos = (np.abs(freq_target - freq)).argmin() # get the index in pos dom of freq_target of the peak
    idx_pos2 = (np.abs(freq_target + freq)).argmin() # neg dom
    # peak_idx = np.argmin(freqs - freq)
    ft_target[idx_pos] = np.abs(signal_f[len(signal) // 2:][peak_idx]) # set amplitude pos dom
    ft_target[idx_pos2] = ft_target[idx_pos] # neg dom

signal_target = np.int16(np.fft.ifft(ft_target.real).real) # FT back to time domain
### END SOLUTION

writeWave('overtone.wav',signal_target)
ipd.Audio('./overtone.wav')

## 3.5 (1 point) — Piano Sound Synthesis - Sound decay

To make the synthesized sound more realistic, include a time-dependent amplitude decay. Model the envelope with an exponential function

$$
e(t) = e^{-\alpha t},
$$

where we set $\alpha = 1$ as the decay rate.


In [ ]:
### BEGIN SOLUTION
signal_target2 = signal_target * np.exp(-t_target)
plt.plot(t_target, signal_target2)
plt.show()
### END SOLUTION

writeWave("overtoneDecay.wav", signal_target2)
ipd.Audio("./overtoneDecay.wav")

## 3.6 (1 point)

Consider the spectrum of the G note, which is the "dominant" note to C. Why do the two notes sound pleasant together? Compare the spectra of the two notes.

In [ ]:
ipd.Audio('./G3.wav')

In [ ]:
signal, sf = readWave("./C3_131_Hz.wav")
signalG, sf = readWave("./G3.wav")
writeWave("./C+G.wav", signalG / 2 + signal / 2)
ipd.Audio("./C+G.wav")

In [ ]:
### BEGIN SOLUTION
plt.plot(freqs[len(signal) // 2 :], np.abs(signal_f[len(signal) // 2 :]))
signalG_f = np.fft.fftshift(np.fft.fft(signalG))
plt.plot(freqs[len(signal) // 2 :], np.abs(signalG_f[len(signal) // 2 :]))
plt.xscale("log")
plt.yscale("log")
plt.ylim(1e5)
plt.xlim(100, 2000)
plt.xlabel("f / Hz")
plt.ylabel("amplitude / a.u.")
plt.show()
### END SOLUTION

### Solution:

From the plot we see that $2f_G = 3f_C$. Thus overtones match for all $2nf_G = 3nf_C$, $n\in \mathbb{N}$. Overlapping overtones with "space" in between in the spectrum sounds good, apparently. The reason for this is neurological, cultural, or both, but we just accept that it's true for now.

If we want different frequencies with lots of overlapping overtones and space in between, we need 
$$nf_1 = mf_2, \quad \text{with} \quad n, m \in \mathbb{N}$$
Lower $n$ and $m$ will naturally give us more overlap of overtones. We could obviously choose the higher frequency to be double that of the lower (one octave apart), i.e. $n=1, m=2$. But that would sound very much like playing just the lower note (actually in musical terminology it is considered the same note!), since its overtones would overlap perfectly with the fundamental and overtones of the higher note. Thus we want the higher note to be less than an octave apart from the lower. The smallest numbers we can choose that fulfill the criteria are $n=2$ and $m=3$.
